# 053 — Design-group collapse fragilities in AvgSA (group-aware replacement for `014`)

Re-express the IDA-based **collapse fragilities** of the EC8-gen2 case-study CBFs from
their native **SA(T1)** into **AvgSA_03** (0–3 s) and **AvgSA_06** (0–6 s), exactly as
notebook `014` does — but reading the **per-design-group** FEMA P695 IDAs written by
`052` instead of the 60 per-site IDAs.

Notebook `051` showed the 120 (site, storey) designs collapse to **51 unique designs**
(25 × 3s, 26 × 5s). The IDA depends only on the structure (the 22 FEMA P695 records are
fixed), so one IDA per group is enough: `052` runs
`D:/08_wp1_fixed_record_sets/group_{n}s_{gid}/{n}s/mdof/ida_femap695/`, and this notebook

1. converts **each group's** IDA once (SA, AvgSA_03, AvgSA_06), caching the result under
   `data_processed/09_structure_fragility_curves/wp1_design_groups/group_{n}s_{gid}/`, and
2. **fans that fragility back out** to every member site as
   `wp1_casestudy_sites/site_{ii}/{tag}_ida_femap695_collapsefragility_{im}.json`
   — exactly the filenames `014` wrote.

Step 2 is what makes this a drop-in replacement: `054` (stripe IMLs from the
representative's AvgSA_03 fragility) and `061`
(`msa_ida_hypothesis_testing.load_site_fragility_curves`) read those per-site paths and
need no change at all.

`014` is left in place for now but nothing here depends on it: this notebook does not read
it, does not delete it, and does not touch its figures or its summary CSV. It *does*
rewrite the per-site fragility JSONs, which are `014`'s output as much as `053`'s.

## Why the output is the same — a one-off check, not a fixture

The conversion machinery is identical to `014` (`convert_ida_collapse_imls` +
`fragility_from_ida`, `g_factor = 1/GRAVITY`). The only question is whether *one* IDA per
group may stand in for the per-site IDAs, and **§2, §3 and §6 answer it once**:

- **§3 — from the per-site fragilities already on disk, no IDA results needed.** Within
  every 3s design group, the 60 fragilities `014` wrote agree to machine precision
  (largest relative spread ~1e-16 in both median and dispersion, all three IMs). The sites
  in a group really do carry one and the same fragility, so collapsing them loses nothing.
- **§6 — against the baseline read in §2.** After the fan-out, every per-site file is
  compared with the value it replaced and the largest relative difference is reported.

Those three sections are gated on `COMPARE_WITH_NB014` and are **meant to be deleted**
once they have passed — they are only meaningful on the first run, while the files on disk
are still `014`'s. Everything from §4 on stands alone.

There is a third, free check while `052`'s migration is fresh: for the 3s groups whose
results were *migrated* (a byte copy of the representative site's IDA folder),
`cache_utils.fingerprint` is content-based, so the group conversion sees exactly the hashes
`014`'s manifests recorded — those files come back as `cached`.

## Inputs

- `D:/08_wp1_fixed_record_sets/group_{n}s_{gid}/{n}s/mdof/ida_femap695/` — the group IDAs
  (`ida_results.pickle`, `record_logs.json`, `record_*/ida_result_record_*.pickle`), `052`.
- `unique_structural_designs.csv` (nb `051`) — the (site, storey) → group table.
- `results/06_group_ida_femap695/group_folder_map.csv` (nb `052`) — cross-checked, not required.
- `ec8_gen2_site_specific_cbfs.pickle` (nb `011`) — structure names and `Vb_coeff`.

## Outputs

| Path | Content |
|---|---|
| `…/09_structure_fragility_curves/wp1_design_groups/group_{n}s_{gid}/group_{n}s_{gid}_ida_femap695_collapsefragility_{im}.json` | the group conversion (+ manifest) |
| `…/09_structure_fragility_curves/wp1_casestudy_sites/site_{ii}/{tag}_ida_femap695_collapsefragility_{im}.json` | the per-site copy `014` used to write (+ manifest) |
| `…/wp1_design_groups/group_femap695_collapsefragility_summary.csv` | group-level median/β table |
| `…/wp1_casestudy_sites/site_femap695_collapsefragility_summary_from_groups.csv` | site-level table (`014`'s CSV is left alone unless `OVERWRITE_SITE_SUMMARY = True`) |
| `results/06_group_ida_femap695/*.png` | the three figures (`014`'s PNGs are left alone) |

> **Run order.** `011` → `051` → `052` (+ the IDA runs) → **053** → `054`, `060`, `061`.
> The IDA results live on `D:` — run this on the workstation. Without the drive the
> notebook still runs §1–§3 (the grouping check) and then reports that nothing could be
> converted.

## Open questions, and the assumption taken for each

Written down for review — every one of these was decided here, not asked.

| # | Question | Assumption taken | If it is wrong |
|---|---|---|---|
| 1 | Should `053` write per-**group** fragility files, or per-**site** ones? | **Both.** The group file is the cached artefact; it is then copied to every member site under the exact `014` filename. Otherwise `054`/`061` would have to learn about groups. | Set `FANOUT_TO_SITES = False` to write only the group files, and teach the downstream notebooks the group map. |
| 2 | May `053` overwrite `014`'s per-site JSONs? | **Yes**, in place and with no backup kept. They are the files `054`/`061` read, so they have to be current, and `053` reproduces them exactly (§6). | Point `OUT_ROOT` at a scratch folder for a dry run first. |
| 2b | Should the comparison against `014` stay in the notebook? | **No** — §2/§3/§6 are gated on `COMPARE_WITH_NB014` and are there for the first run only. Once they pass, set it `False` or delete those three sections; they read the per-site files as a "baseline", which after the first `053` run is just `053`'s own previous output. | — |
| 3 | …and `014`'s summary CSV and PNGs? | **No.** The CSV goes to `…_summary_from_groups.csv` and the figures to `results/06_group_ida_femap695/`, so `014`'s outputs survive for comparison. | Set `OVERWRITE_SITE_SUMMARY = True` once you are happy. |
| 4 | Which storey counts? | **3 and 5.** `014` did 3s only (the 5s IDAs had not been run); `052` sets up both, so both are converted here. 5s simply has no `014` baseline for §6 to compare against. | Edit `STOREYS`. |
| 5 | Where does the group's `T1` come from? | The group IDA's own record logs (`intensity_measure=…period=…`), exactly as in `014`. One structure per group, so one `T1`. | — |
| 6 | What should the per-site copies fingerprint? | The **group fragility JSON** (plus group name / site / storeys), not the raw IDA — a site file is stale exactly when its group conversion changed. This is a different provenance chain from `014`'s (which named `ida_results.pickle`), so the first run of `053` rewrites every per-site file even where the value is unchanged. | — |
| 7 | Non-representative member sites still have their own per-site IDA folders under `D:/07_wp1_casestudy_sites`. Should those be used at all? | **No** — ignored entirely. §3 shows they carry the same fragility as their group. | — |
| 8 | `052`'s group map currently lists `needs_run = True` for every group. Should a missing group IDA be an error? | **No** — reported and skipped, as `014` did for sites without IDA output, so the notebook stays useful while the runs trickle in. | Set `STRICT = True` to raise instead. |
| 9 | Should `053` also re-derive `Vb_coeff`? | **No** — read from the `011` site dataset, as `014` did. The plots colour each *site* by its own `Vb_coeff`; sites in a group share a design, so they share it anyway. | — |

## 0. Setup & parameters

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import re
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize
from scipy.stats import lognorm

from phd_project.config import config
from phd_project.scripts.cache_utils import fingerprint, json_load_or_compute
from phd_project.scripts.WP1_ground_motion_set.design_groups import load_design_groups
from standes.analysis.ida import (
    collect_ida_results, collect_record_logs, get_collapse_iml,
    convert_ida_collapse_imls
)
from standes.fragility_curves import fragility_from_ida
from standes.intensitymeasures import SpectralAcceleration, avgsa_03, avgsa_06
from standes.groundmotion import load_ground_motion_from_json

cfg = config.load_config()

In [3]:
# ----------------------------------------------------------------------------
# PARAMETERS
# ----------------------------------------------------------------------------
# Root of the per-DESIGN-GROUP analysis folders written by nb 052. Per group the FEMA
# P695 IDA lives at GROUP_ROOT/group_{n}s_{gid}/{n}s/mdof/ida_femap695/.
GROUP_ROOT = Path(cfg["analysis_data"]["wp1_fixed_record_sets"])

# nb 052's group -> member-site map. Cross-checked against nb 051's table; not required.
GROUP_MAP_PATH = Path(cfg["results"]["group_ida_femap695"]) / "group_folder_map.csv"

# Site dataset (nb 011): structure names and the design base-shear coefficient Vb_coeff.
SITE_DATASET = Path(cfg["proc_data"]["site_specific_dataset"])

STOREYS = [3, 5]

# --- outputs ---
# The group conversion (the cached artefact).
GROUP_OUT_ROOT = Path(cfg["proc_data"]["wp1_groups_fragility_curves"])
# The per-site copies - the same files nb 014 writes, read by nb 054 / 061.
OUT_ROOT = Path(cfg["proc_data"]["wp1_sites_fragility_curves"])
# Figures. Deliberately NOT nb 014's folder, so its PNGs survive.
FIG_ROOT = Path(cfg["results"]["group_ida_femap695"])

for _d in (GROUP_OUT_ROOT, OUT_ROOT, FIG_ROOT):
    _d.mkdir(parents=True, exist_ok=True)

# Copy the group fragility out to every member site under nb 014's filenames. False
# writes only the group-level files (nothing downstream would then find its fragility).
FANOUT_TO_SITES = True

# Sections 3 and 6 - the ONE-OFF validation against nb 014. They read whatever per-site
# fragilities are on disk BEFORE this run overwrites them and check that 053 reproduces
# them. Only meaningful while those files are still nb 014's: after 053 has written them
# once, the "baseline" is 053's own previous output and the check is self-referential.
# Turn this off (and delete sections 3 and 6) once the comparison has passed - nothing
# else in the notebook depends on it.
COMPARE_WITH_NB014 = True

# Overwrite nb 014's site-level summary CSV. False writes ..._from_groups.csv instead,
# leaving nb 014's output intact for comparison. Flip once 053 is trusted.
OVERWRITE_SITE_SUMMARY = False

# Raise instead of skipping when a group has no IDA output on disk.
STRICT = False

# Force a full re-conversion even when the provenance manifest still matches.
FORCE_RECOMPUTE = False

# IML units of the IDA [mm/s^2]. Every fragility written here is rescaled by 1/GRAVITY
# so all medians and empirical curves are a fraction of g. Same as nb 014.
GRAVITY = 9810.0

# Target intensity measures (SA(T1) is handled separately as the native IM).
AVGSA_IMS = {"AvgSA_03": avgsa_03(), "AvgSA_06": avgsa_06()}
IM_ORDER = ["SA", "AvgSA_03", "AvgSA_06"]  # panel / column order

IDA_TAG = "ida_femap695"

# Summary CSVs. Derived from the roots above rather than read straight from the config,
# so pointing OUT_ROOT / GROUP_OUT_ROOT at a scratch folder keeps a dry run entirely
# self-contained. The filenames are the config's.
GROUP_SUMMARY_CSV = GROUP_OUT_ROOT / Path(
    cfg["proc_data"]["wp1_groups_femap695_fc_summary"]).name
_site_csv = Path(cfg["proc_data"]["wp1_sites_femap695_fc_summary"])
SITE_SUMMARY_CSV = OUT_ROOT / (_site_csv.name if OVERWRITE_SITE_SUMMARY
                               else _site_csv.stem + "_from_groups.csv")

print(f"GROUP_ROOT     = {GROUP_ROOT}")
print(f"GROUP_OUT_ROOT = {GROUP_OUT_ROOT}")
print(f"OUT_ROOT       = {OUT_ROOT}  (fan-out: {FANOUT_TO_SITES})")
print(f"FIG_ROOT       = {FIG_ROOT}")

GROUP_ROOT     = D:\08_wp1_fixed_record_sets
GROUP_OUT_ROOT = C:\Users\clemettn\Documents\phd\data_processed\09_structure_fragility_curves\wp1_design_groups
OUT_ROOT       = C:\Users\clemettn\Documents\phd\data_processed\09_structure_fragility_curves\wp1_casestudy_sites  (fan-out: True)
FIG_ROOT       = C:\Users\clemettn\Documents\phd\results\06_group_ida_femap695


## 1. Design groups and their IDA folders

One row per design group from `051` (`is_representative == True`), with its member sites.
The folder convention is `052`'s: `GROUP_ROOT/group_{n}s_{gid:02d}/{n}s/mdof/ida_femap695`.

Three checks run here:

- every group is non-empty and every (site, storey) belongs to exactly one group;
- the member lists cover every site of each storey count in `STOREYS`;
- the derived folders agree with `052`'s `group_folder_map.csv` when that file exists (it
  is written in-repo, so this check works with the analysis drive detached).

In [4]:
groups_df = load_design_groups(cfg)
groups_df = groups_df[groups_df["storeys"].isin(STOREYS)].copy()

groups = (groups_df[groups_df["is_representative"]]
          .loc[:, ["storeys", "group_id", "representative_site", "representative_tag",
                   "n_sites_in_group"]]
          .sort_values(["storeys", "group_id"])
          .reset_index(drop=True))

members = (groups_df.groupby(["storeys", "group_id"])["site"]
           .apply(lambda s: sorted(int(x) for x in s)))


def group_folder_name(n: int, gid: int) -> str:
    """Folder for a design group: group_3s_00, group_5s_07, ... (nb 052's convention)."""
    return f"group_{int(n)}s_{int(gid):02d}"


def group_ida_folder(n: int, gid: int, ida_tag: str = IDA_TAG) -> Path:
    return GROUP_ROOT / group_folder_name(n, gid) / f"{int(n)}s" / "mdof" / ida_tag


# --- sanity checks on the grouping itself ---
dupes = int(groups_df.duplicated(subset=["storeys", "site"]).sum())
assert dupes == 0, f"{dupes} (site, storey) pairs appear in more than one group"
assert (groups["n_sites_in_group"] > 0).all(), "a group has no member sites"

for n in STOREYS:
    all_sites = sorted(groups_df.loc[groups_df["storeys"] == n, "site"].astype(int))
    from_members = sorted(s for gid in groups.loc[groups["storeys"] == n, "group_id"]
                          for s in members[(n, int(gid))])
    assert from_members == all_sites, f"{n}s: member lists do not cover every site once"
    print(f"{n}s: {int((groups['storeys'] == n).sum()):2d} design groups covering "
          f"{len(all_sites)} sites (site {min(all_sites)}..{max(all_sites)})")

groups["ida_folder"] = [group_ida_folder(r.storeys, r.group_id)
                        for r in groups.itertuples()]
groups["ida_exists"] = [p.is_dir() for p in groups["ida_folder"]]
print(f"\n{int(groups['ida_exists'].sum())}/{len(groups)} group(s) have IDA output on disk")
if not GROUP_ROOT.exists():
    print(f"NOTE: {GROUP_ROOT} is not reachable - the analysis drive is detached, so "
          f"nothing can be converted in this run.")
groups.head()

3s: 25 design groups covering 60 sites (site 0..59)
5s: 26 design groups covering 60 sites (site 0..59)

0/51 group(s) have IDA output on disk
NOTE: D:\08_wp1_fixed_record_sets is not reachable - the analysis drive is detached, so nothing can be converted in this run.


,storeys,group_id,representative_site,representative_tag,n_sites_in_group,ida_folder,ida_exists
0,3,0,0,3s_cbf_dc2_site0,8,D:\08_wp1_fixed_record_sets\group_3s_00\3s\mdo...,False
1,3,1,1,3s_cbf_dc2_site1,6,D:\08_wp1_fixed_record_sets\group_3s_01\3s\mdo...,False
2,3,2,3,3s_cbf_dc2_site3,1,D:\08_wp1_fixed_record_sets\group_3s_02\3s\mdo...,False
3,3,3,7,3s_cbf_dc2_site7,5,D:\08_wp1_fixed_record_sets\group_3s_03\3s\mdo...,False
4,3,4,9,3s_cbf_dc2_site9,3,D:\08_wp1_fixed_record_sets\group_3s_04\3s\mdo...,False


In [5]:
# --- cross-check the derived folders against nb 052's group map ---
if not GROUP_MAP_PATH.is_file():
    print(f"no group map at {GROUP_MAP_PATH} (nb 052 not run here) - skipping cross-check")
else:
    gmap = pd.read_csv(GROUP_MAP_PATH)
    gmap = gmap[gmap["storeys"].isin(STOREYS)]
    mapped = {(int(r.storeys), int(r.group_id)): r for r in gmap.itertuples()}
    mismatch = []
    for r in groups.itertuples():
        key = (int(r.storeys), int(r.group_id))
        m = mapped.get(key)
        if m is None:
            mismatch.append(f"{group_folder_name(*key)}: absent from the group map")
            continue
        # nb 052 stores the mdof folder; the IDA folder is that + the ida_tag.
        expect = (Path(m.folder) / m.ida_tag).as_posix().lower()
        if Path(r.ida_folder).as_posix().lower() != expect:
            mismatch.append(f"{group_folder_name(*key)}: {r.ida_folder} != {expect}")
        if int(m.representative_site) != int(r.representative_site):
            mismatch.append(f"{group_folder_name(*key)}: representative site differs")
        if sorted(int(s) for s in str(m.sites).split()) != members[key]:
            mismatch.append(f"{group_folder_name(*key)}: member sites differ")
    if mismatch:
        raise AssertionError("group map disagrees with nb 051's table:\n  "
                             + "\n  ".join(mismatch))
    print(f"group map agrees with nb 051 for all {len(groups)} group(s): folders, "
          f"representatives and member lists")

group map agrees with nb 051 for all 51 group(s): folders, representatives and member lists


In [6]:
# --- structure names and design base-shear coefficients (nb 011), for the plots ---
designs = pd.read_pickle(SITE_DATASET)
designs = designs[designs["n_storeys"].isin(STOREYS)].copy()

NAME = {(int(r.n_storeys), int(r.site_idx)): r.structure_name for r in
        designs.rename(columns={"name": "structure_name"}).itertuples()}
VB_COEFF = {(int(r.n_storeys), int(r.site_idx)): float(r.Vb_coeff)
            for r in designs.itertuples()}


def structure_tag(n: int, site: int) -> str:
    """The per-site structure tag, straight from the nb 011 dataset.

    Falls back to the naming convention ("{n}s_cbf_dc2_site{ii}", as hard-coded in
    msa_ida_hypothesis_testing.structure_tag) for a structure missing from it.
    """
    return NAME.get((int(n), int(site)), f"{int(n)}s_cbf_dc2_site{int(site)}")


for n in STOREYS:
    absent = [s for s in sorted(groups_df.loc[groups_df["storeys"] == n, "site"].astype(int))
              if (n, s) not in NAME]
    if absent:
        print(f"WARNING: {n}s: {len(absent)} site(s) are not in the nb 011 dataset, so "
              f"their tags fall back to the naming convention: {absent}")
print(f"{len(NAME)} structures with a tag and Vb_coeff from {SITE_DATASET.name}")

120 structures with a tag and Vb_coeff from ec8_gen2_site_specific_cbfs.pickle


## 2. Read the existing per-site fragilities as a baseline

*Part of the one-off validation — the whole of §2, §3 and §6 can be deleted once it has
passed, and is skipped entirely when `COMPARE_WITH_NB014 = False`.*

The fan-out in §5 rewrites `site_{ii}/{tag}_ida_femap695_collapsefragility_{im}.json`, so
whatever is there now is first read into memory as
`BASELINE[(n, site, im)] -> {median, dispersion, efc, …}`. Nothing is copied and nothing
is written.

Read it **before** `053` has ever run: the files are then still `014`'s, and §6 is a real
comparison. Afterwards they are `053`'s own previous output and §6 only confirms the run
was reproducible.

In [7]:
def site_fragility_path(root: Path, n: int, site: int, im: str,
                        ida_tag: str = IDA_TAG) -> Path:
    tag = structure_tag(n, site)
    return root / f"site_{int(site)}" / f"{tag}_{ida_tag}_collapsefragility_{im}.json"


BASELINE: dict[tuple[int, int, str], dict] = {}

if not COMPARE_WITH_NB014:
    print("COMPARE_WITH_NB014 is False - no baseline read; sections 3 and 6 will skip")
else:
    for n in STOREYS:
        for site in sorted(groups_df.loc[groups_df["storeys"] == n, "site"].astype(int)):
            for im in IM_ORDER:
                fp = site_fragility_path(OUT_ROOT, n, site, im)
                if fp.is_file():
                    BASELINE[(n, site, im)] = json.loads(fp.read_text())

    by_storey = {n: len({k[1] for k in BASELINE if k[0] == n}) for n in STOREYS}
    print(f"baseline: {len(BASELINE)} existing fragility file(s) read "
          f"({', '.join(f'{n}s: {c} sites' for n, c in by_storey.items())})")

baseline: 180 existing fragility file(s) read (3s: 60 sites, 5s: 0 sites)


## 3. Does one IDA per group really stand in for its member sites?

*One-off validation — delete with §2 and §6 once it has passed.*

This is the substantive question behind the whole notebook, and it can be answered **from
the per-site fragilities already on disk** — no IDA results, no analysis drive.

If the sites in a group share a byte-identical design (which is how `051` defines a
group), their per-site IDAs are the same analysis of the same model, so `014`'s
fragilities for them must already be identical. Below, for each group and each IM, the
relative spread across member sites

$$\frac{\max_i x_i - \min_i x_i}{\bar{x}}$$

is computed for both the median $\theta$ and the dispersion $\beta$. Anything beyond
floating-point noise (`TOL`) would mean the grouping is too coarse and this notebook must
not replace `014`.

*Result when this notebook was written (3s, all 25 groups, all three IMs): the largest
spread was `1.2e-16` — identical to the last bit.*

In [8]:
TOL = 1e-9      # relative spread beyond this is a real disagreement, not fp noise

spread_rows = []
for r in groups.itertuples():
    n, gid = int(r.storeys), int(r.group_id)
    for im in IM_ORDER:
        vals = [BASELINE[(n, s, im)] for s in members[(n, gid)]
                if (n, s, im) in BASELINE]
        if len(vals) < 2:
            continue
        med = np.array([v["median"] for v in vals])
        dis = np.array([v["dispersion"] for v in vals])
        spread_rows.append({
            "group": group_folder_name(n, gid), "storeys": n, "im": im,
            "n_sites_with_014": len(vals),
            "median_spread": float((med.max() - med.min()) / med.mean()),
            "dispersion_spread": float((dis.max() - dis.min()) / dis.mean()),
        })

spread = pd.DataFrame(spread_rows)
if spread.empty:
    print("skipped: no baseline with 2+ member sites on disk "
          f"(COMPARE_WITH_NB014 = {COMPARE_WITH_NB014}). This check needs the "
          "per-site fragilities nb 014 wrote, which exist for 3s only.")
else:
    worst = spread[["median_spread", "dispersion_spread"]].max()
    print(f"{len(spread)} (group, IM) combinations with 2+ member sites")
    print(f"worst within-group spread:  median {worst['median_spread']:.2e}, "
          f"dispersion {worst['dispersion_spread']:.2e}   (tolerance {TOL:.0e})")
    bad = spread[(spread["median_spread"] > TOL) | (spread["dispersion_spread"] > TOL)]
    if bad.empty:
        print("PASS - within every design group, nb 014's per-site fragilities are "
              "identical to machine precision, so one IDA per group loses nothing.")
    else:
        print(f"FAIL - {len(bad)} group(s) disagree beyond the tolerance:")
        print(bad.to_string(index=False))

42 (group, IM) combinations with 2+ member sites
worst within-group spread:  median 1.19e-16, dispersion 1.70e-16   (tolerance 1e-09)
PASS - within every design group, nb 014's per-site fragilities are identical to machine precision, so one IDA per group loses nothing.


## 4. Convert each design group

For each group: read its IDA results and per-record logs, recover the design period `T1`
(and hence the native `SpectralAcceleration(T1)`) from the IDA intensity measure, load the
ground-motion records, and convert the collapse IMLs into each AvgSA.

Identical to `014` section 3, only keyed by group:

- all three curves are refit from the per-record collapse IMLs with `fragility_from_ida`,
  so they come out in the same units; `g_factor = 1/GRAVITY` puts medians and empirical
  curves in **g** and leaves the dimensionless dispersion alone;
- the SA fit reproduces the IDA's own `collapse_fragility.json` (`get_collapse_iml`
  returns the same `spline[-1, 1]` that `CollapseFragilityConstructorIDA` fits), rescaled;
- `convert_ida_collapse_imls` keeps its default `gravity_factor=1`: that factor scales both
  intensity measures inside `convert_im` and cancels out of the ratio, so it cannot change
  the output units — only `g_factor` can.

Caching is `cache_utils.json_load_or_compute` against the group's `ida_results.pickle` /
`record_logs.json`, so a re-run only re-converts groups whose IDA changed. The IDA results
are loaded **lazily**: a group whose three JSONs are all still valid never touches `D:`.

In [9]:
def parse_T1(logs: dict) -> float:
    im_str = next(iter(logs.values()))["intensity_measure"]
    return float(re.search(r"period=([0-9.eE+-]+)", im_str).group(1))


def convert_group(n: int, gid: int, ida_tag: str = IDA_TAG,
                  force: bool = FORCE_RECOMPUTE):
    """Convert one design group's IDA into the three IMs. Returns (fragilities, info)."""
    name = group_folder_name(n, gid)
    fol = group_ida_folder(n, gid, ida_tag)
    if not fol.is_dir():
        if STRICT:
            raise FileNotFoundError(f"{name}: no IDA output at {fol}")
        print(f"[skip] {name}: no IDA output at {fol}")
        return None, {}

    out_dir = GROUP_OUT_ROOT / name
    out_dir.mkdir(parents=True, exist_ok=True)
    ida_results_fp = fol / "ida_results.pickle"
    record_logs_fp = fol / "record_logs.json"

    loaded: dict = {}

    def _load():
        """Read the IDA once, and only if something actually has to be converted."""
        if not loaded:
            ida_results = collect_ida_results(fol)
            logs = collect_record_logs(fol)
            loaded["ida_results"] = ida_results
            loaded["logs"] = logs
            loaded["original_im"] = SpectralAcceleration(parse_T1(logs))
            loaded["gm_records"] = {
                rec: load_ground_motion_from_json(Path(logs[rec]["record_path"]))
                for rec in ida_results}
        return loaded

    out, statuses = {}, {}

    # --- native SA(T1): refit the IDA's own collapse IMLs, rescaled to g ---
    def _fit_sa():
        L = _load()
        imls = [get_collapse_iml(r) for r in L["ida_results"].values()]
        return fragility_from_ida(imls, im=L["original_im"],
                                  g_factor=1 / GRAVITY).asdict()

    out["SA"], statuses["SA"] = json_load_or_compute(
        out_dir / f"{name}_{ida_tag}_collapsefragility_SA.json",
        fingerprint(ida_results=ida_results_fp), _fit_sa, force=force,
        input_paths={"ida_results": ida_results_fp})

    # --- AvgSA conversions ---
    for im_name, new_im in AVGSA_IMS.items():
        def _convert(new_im=new_im):
            L = _load()
            conv = convert_ida_collapse_imls(L["ida_results"], L["gm_records"],
                                             L["original_im"], new_im)
            return fragility_from_ida(conv.values(), im=new_im,
                                      g_factor=1 / GRAVITY).asdict()

        out[im_name], statuses[im_name] = json_load_or_compute(
            out_dir / f"{name}_{ida_tag}_collapsefragility_{im_name}.json",
            fingerprint(ida_results=ida_results_fp, record_logs=record_logs_fp,
                        im_periods=np.asarray(new_im.periods)),
            _convert, force=force,
            input_paths={"ida_results": ida_results_fp, "record_logs": record_logs_fp})

    info = {"n_records": len(loaded["ida_results"]) if loaded else None,
            "T1": float(loaded["original_im"].period) if loaded else None}
    detail = (f", {info['n_records']} records, T1={info['T1']:.3f}s"
              if info["n_records"] else " (all cached, IDA not read)")
    print(f"[ok]   {name}: " + ", ".join(f"{k}={v}" for k, v in statuses.items()) + detail)
    return out, info


group_results: dict[tuple[int, int], dict] = {}
group_info: dict[tuple[int, int], dict] = {}

for r in groups.itertuples():
    key = (int(r.storeys), int(r.group_id))
    res, info = convert_group(*key)
    if res is not None:
        group_results[key] = res
        group_info[key] = info

print(f"\nconverted {len(group_results)}/{len(groups)} design groups -> {GROUP_OUT_ROOT}")

[skip] group_3s_00: no IDA output at D:\08_wp1_fixed_record_sets\group_3s_00\3s\mdof\ida_femap695
[skip] group_3s_01: no IDA output at D:\08_wp1_fixed_record_sets\group_3s_01\3s\mdof\ida_femap695
[skip] group_3s_02: no IDA output at D:\08_wp1_fixed_record_sets\group_3s_02\3s\mdof\ida_femap695
[skip] group_3s_03: no IDA output at D:\08_wp1_fixed_record_sets\group_3s_03\3s\mdof\ida_femap695
[skip] group_3s_04: no IDA output at D:\08_wp1_fixed_record_sets\group_3s_04\3s\mdof\ida_femap695
[skip] group_3s_05: no IDA output at D:\08_wp1_fixed_record_sets\group_3s_05\3s\mdof\ida_femap695
[skip] group_3s_06: no IDA output at D:\08_wp1_fixed_record_sets\group_3s_06\3s\mdof\ida_femap695
[skip] group_3s_07: no IDA output at D:\08_wp1_fixed_record_sets\group_3s_07\3s\mdof\ida_femap695
[skip] group_3s_08: no IDA output at D:\08_wp1_fixed_record_sets\group_3s_08\3s\mdof\ida_femap695
[skip] group_3s_09: no IDA output at D:\08_wp1_fixed_record_sets\group_3s_09\3s\mdof\ida_femap695
[skip] group_3s_10: 

In [10]:
# The FEMA P695 far-field set is 22 records; a group converted from fewer means its IDA
# is incomplete (or some record folders failed) and its fragility is fitted from a
# smaller sample than nb 014's. Groups served from cache report no count - nothing was
# read - so they are listed separately rather than flagged.
N_EXPECTED_RECORDS = 22

short = {k: i["n_records"] for k, i in group_info.items()
         if i.get("n_records") and i["n_records"] != N_EXPECTED_RECORDS}
cached_only = [k for k, i in group_info.items() if not i.get("n_records")]
if short:
    print(f"WARNING: {len(short)} group(s) converted from != {N_EXPECTED_RECORDS} records:")
    for k, c in sorted(short.items()):
        print(f"  {group_folder_name(*k)}: {c} records")
elif group_results:
    print(f"every freshly converted group used {N_EXPECTED_RECORDS} records")
print(f"{len(cached_only)} group(s) served entirely from cache (IDA not read)")

0 group(s) served entirely from cache (IDA not read)


## 5. Fan the group fragility out to its member sites

Each group's three fragilities are copied to **every** member site under exactly the
filenames `014` wrote, so `054` (which inverts
`{rep_tag}_ida_femap695_collapsefragility_AvgSA_03.json`) and `061`
(`load_site_fragility_curves`) keep working untouched.

The per-site manifest fingerprints the **group fragility JSON**, not the raw IDA — a site
file is stale exactly when its group's conversion changed. That is a different provenance
chain from `014`'s, so the first run rewrites every per-site file even where the number is
unchanged; §6 is what confirms the numbers did not move.

In [11]:
def group_fragility_path(n: int, gid: int, im: str, ida_tag: str = IDA_TAG) -> Path:
    name = group_folder_name(n, gid)
    return GROUP_OUT_ROOT / name / f"{name}_{ida_tag}_collapsefragility_{im}.json"


site_results: dict[tuple[int, int], dict] = {}   # (storeys, site) -> {im: fragility}
site_group: dict[tuple[int, int], tuple[int, int]] = {}
fanout_status = {"cached": 0, "computed": 0}

if not FANOUT_TO_SITES:
    print("FANOUT_TO_SITES is False - only the group-level files were written")
else:
    for (n, gid), res in sorted(group_results.items()):
        name = group_folder_name(n, gid)
        for site in members[(n, gid)]:
            out = {}
            for im in IM_ORDER:
                src = group_fragility_path(n, gid, im)
                fp = site_fragility_path(OUT_ROOT, n, site, im)
                inputs = fingerprint(group_fragility=src, group=name,
                                     site=int(site), storeys=int(n))

                def _copy(im=im):
                    return res[im]

                out[im], status = json_load_or_compute(
                    fp, inputs, _copy, force=FORCE_RECOMPUTE,
                    input_paths={"group_fragility": src})
                fanout_status[status] += 1
            site_results[(n, site)] = out
            site_group[(n, site)] = (n, gid)
        print(f"[fanout] {name} -> {len(members[(n, gid)])} site(s): "
              f"{', '.join(f'site_{s}' for s in members[(n, gid)])}")

    print(f"\n{len(site_results)} structure(s) written to {OUT_ROOT} "
          f"({fanout_status['computed']} file(s) written, "
          f"{fanout_status['cached']} already current)")


0 structure(s) written to C:\Users\clemettn\Documents\phd\data_processed\09_structure_fragility_curves\wp1_casestudy_sites (0 file(s) written, 0 already current)


## 6. Verification against nb 014

*One-off validation — delete with §2 and §3 once it has passed.*

Every per-site fragility just written is compared with the baseline read in §2 —
median, dispersion and the full empirical curve `efc` — as relative differences

$$\left|\frac{x_{053} - x_{014}}{x_{014}}\right|.$$

Two families of structure are separated, because only one of them is *expected* to be
bit-identical for a trivial reason:

- **representative sites** — `052` migrated their IDA folder as a byte copy, so the content
  hashes match `014`'s manifests exactly and nothing should move at all;
- **other member sites** — converted here from the representative's IDA rather than from
  their own. §3 already showed the two give the same fragility; this is the direct check.

Anything above `VERIFY_TOL` is listed in full.

In [12]:
VERIFY_TOL = 1e-9      # relative difference above this is a real change

REPS = {(int(r.storeys), int(r.representative_site)) for r in groups.itertuples()}
DIFF_COLS = ["d_median", "d_dispersion", "d_efc"]


def _rel(a, b) -> float:
    """Largest elementwise |a-b|/|b| (|b| floored at 1 where b == 0)."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if a.shape != b.shape:
        return float("inf")
    denom = np.where(np.abs(b) > 0, np.abs(b), 1.0)
    return float(np.max(np.abs(a - b) / denom))


cmp_rows = []
for (n, site), out in sorted(site_results.items()):
    for im in IM_ORDER:
        base = BASELINE.get((n, site, im))
        if base is None:
            continue
        new = out[im]
        cmp_rows.append({
            "storeys": n, "site": site, "im": im,
            "is_representative": (n, site) in REPS,
            "d_median": abs(new["median"] - base["median"]) / abs(base["median"]),
            "d_dispersion": abs(new["dispersion"] - base["dispersion"]) / abs(base["dispersion"]),
            "d_efc": _rel(new["efc"], base["efc"]),
            "median_053": new["median"], "median_014": base["median"],
        })

comparison = pd.DataFrame(cmp_rows)

if comparison.empty:
    print("nothing to verify: no structure has both a baseline and a new conversion "
          f"(COMPARE_WITH_NB014 = {COMPARE_WITH_NB014}; also expected with the "
          "analysis drive detached, and for 5s).")
else:
    n_struct = comparison[["storeys", "site"]].drop_duplicates().shape[0]
    print(f"compared {len(comparison)} (structure, IM) pair(s) over {n_struct} structures")
    for label, sub in [("representative sites", comparison[comparison["is_representative"]]),
                       ("other member sites", comparison[~comparison["is_representative"]])]:
        if sub.empty:
            print(f"  {label}: none")
            continue
        w = sub[DIFF_COLS].max()
        print(f"  {label} ({len(sub)} pairs): max |rel diff| median {w['d_median']:.2e}, "
              f"dispersion {w['d_dispersion']:.2e}, efc {w['d_efc']:.2e}")

    bad = comparison[(comparison[DIFF_COLS] > VERIFY_TOL).any(axis=1)]
    if bad.empty:
        print(f"\nPASS - every structure reproduces nb 014 to within {VERIFY_TOL:.0e} "
              f"relative. Notebook 053 is a drop-in replacement.")
    else:
        print(f"\nFAIL - {len(bad)} (structure, IM) pair(s) differ from nb 014:")
        with pd.option_context("display.float_format", lambda v: f"{v:.3e}"):
            print(bad.sort_values("d_median", ascending=False).to_string(index=False))

# Structures nb 014 covered but 053 did not reach (their group IDA is not on disk).
missing_vs_014 = sorted({(n, s) for (n, s, _) in BASELINE} - set(site_results))
if missing_vs_014:
    print(f"\nNOTE: {len(missing_vs_014)} structure(s) have a nb 014 fragility but no new "
          f"conversion: "
          f"{', '.join(f'{n}s/site_{s}' for n, s in missing_vs_014[:20])}"
          + (" ..." if len(missing_vs_014) > 20 else ""))

nothing to verify: no structure has both a baseline and a new conversion (COMPARE_WITH_NB014 = True; also expected with the analysis drive detached, and for 5s).

NOTE: 60 structure(s) have a nb 014 fragility but no new conversion: 3s/site_0, 3s/site_1, 3s/site_2, 3s/site_3, 3s/site_4, 3s/site_5, 3s/site_6, 3s/site_7, 3s/site_8, 3s/site_9, 3s/site_10, 3s/site_11, 3s/site_12, 3s/site_13, 3s/site_14, 3s/site_15, 3s/site_16, 3s/site_17, 3s/site_18, 3s/site_19 ...


### Coverage

Which (storey, site) structures ended up with a fragility. Every site of every storey
count in `STOREYS` should appear exactly once; sites whose group IDA has not been run yet
are listed as missing.

In [13]:
cov_rows = []
for n in STOREYS:
    all_sites = sorted(groups_df.loc[groups_df["storeys"] == n, "site"].astype(int))
    got = [s for s in all_sites if (n, s) in site_results]
    cov_rows.append({"storeys": n, "sites": len(all_sites), "with_fragility": len(got),
                     "missing": len(all_sites) - len(got),
                     "groups_converted": sum(1 for k in group_results if k[0] == n),
                     "groups_total": int((groups["storeys"] == n).sum())})

coverage = pd.DataFrame(cov_rows)
print(coverage.to_string(index=False))
for n in STOREYS:
    miss = [s for s in sorted(groups_df.loc[groups_df["storeys"] == n, "site"].astype(int))
            if (n, s) not in site_results]
    if miss:
        print(f"{n}s missing sites: {miss}")

 storeys  sites  with_fragility  missing  groups_converted  groups_total
       3     60               0       60                 0            25
       5     60               0       60                 0            26
3s missing sites: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]
5s missing sites: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59]


## 7. Summary tables

Both carry the median in **g** and the dimensionless lognormal $\beta$:

- **group level** — one row per design group, with its member sites, written to
  `wp1_design_groups/group_femap695_collapsefragility_summary.csv`;
- **site level** — one row per structure, the same shape as `014`'s CSV. It goes to
  `site_femap695_collapsefragility_summary_from_groups.csv` so `014`'s own file survives;
  set `OVERWRITE_SITE_SUMMARY = True` to write `014`'s path instead.

In [14]:
group_rows = {}
for (n, gid), res in sorted(group_results.items()):
    row = {("info", "storeys"): n,
           ("info", "n_sites"): len(members[(n, gid)]),
           ("info", "sites"): " ".join(str(s) for s in members[(n, gid)])}
    for im in IM_ORDER:
        row[(im, "median")] = res[im]["median"]
        row[(im, "dispersion")] = res[im]["dispersion"]
    group_rows[group_folder_name(n, gid)] = row

group_summary = pd.DataFrame.from_dict(group_rows, orient="index")
if group_summary.empty:
    print("no groups converted - no group summary written")
else:
    group_summary.columns = pd.MultiIndex.from_tuples(group_summary.columns,
                                                      names=["IM", "parameter"])
    group_summary = group_summary[["info"] + IM_ORDER]
    group_summary.index.name = "group"
    with pd.option_context("display.float_format", lambda v: f"{v:.3f}"):
        print(group_summary.to_string())
    group_summary.to_csv(GROUP_SUMMARY_CSV)
    print(f"\nwrote {GROUP_SUMMARY_CSV}")

no groups converted - no group summary written


In [15]:
site_rows = {}
for (n, site), out in sorted(site_results.items()):
    row = {("info", "storeys"): n, ("info", "site"): site,
           ("info", "group"): group_folder_name(*site_group[(n, site)]),
           ("info", "Vb_coeff"): VB_COEFF.get((n, site), np.nan)}
    for im in IM_ORDER:
        row[(im, "median")] = out[im]["median"]
        row[(im, "dispersion")] = out[im]["dispersion"]
    site_rows[structure_tag(n, site)] = row

site_summary = pd.DataFrame.from_dict(site_rows, orient="index")
if site_summary.empty:
    print("no structures converted - no site summary written")
else:
    site_summary.columns = pd.MultiIndex.from_tuples(site_summary.columns,
                                                     names=["IM", "parameter"])
    site_summary = site_summary[["info"] + IM_ORDER]
    site_summary.index.name = "structure"
    site_summary = site_summary.sort_values([("info", "storeys"), ("info", "Vb_coeff")])
    with pd.option_context("display.float_format", lambda v: f"{v:.3f}"):
        print(site_summary.to_string())

    site_summary.to_csv(SITE_SUMMARY_CSV)
    print(f"\nwrote {SITE_SUMMARY_CSV}"
          + ("" if OVERWRITE_SITE_SUMMARY else "  (nb 014's CSV left untouched)"))

no structures converted - no site summary written


## 8. Collapse fragilities by intensity measure

`014`'s figures, redrawn from the fanned-out per-site set — one row of three panels
(**SA(T1)**, **AvgSA_03**, **AvgSA_06**) per storey count. Curves are coloured by the
design base-shear coefficient `Vb_coeff = Vb/Wt` (sequential *viridis*, shared colorbar),
so the collapse-capacity ladder from lightly to heavily designed structures is visible in
every IM. The x-axis is in units of g.

Because every member site of a group now carries the same curve, a group of 8 sites draws
8 coincident lines — the same picture `014` produced, where those 8 curves coincided
anyway.

The PNGs go to `results/06_group_ida_femap695/`; `014`'s figures are left where they
are.

In [16]:
def plot_fragilities(n: int):
    keys = [k for k in sorted(site_results) if k[0] == n]
    if not keys:
        print(f"{n}s: nothing to plot")
        return None
    vbs = np.array([VB_COEFF.get(k, np.nan) for k in keys])
    norm = Normalize(vmin=np.nanmin(vbs), vmax=np.nanmax(vbs))
    cmap = cm.viridis

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharey=True)
    for ax, im in zip(axes, IM_ORDER):
        for k in keys:
            frag = site_results[k][im]
            median_g = frag["median"]
            x = np.linspace(1e-3, median_g * 3.5, 400)
            y = lognorm.cdf(x, s=frag["dispersion"], scale=median_g)
            ax.plot(x, y, lw=1.5, alpha=0.8, color=cmap(norm(VB_COEFF.get(k, np.nan))))
        ax.set_title(im, fontsize=11)
        ax.set_xlabel(f"{im} [g]")
        ax.set_xlim(left=0)
        ax.set_ylim(0, 1)
        ax.grid(True, lw=0.5, alpha=0.4)
    axes[0].set_ylabel("P(collapse | IM)")

    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=axes, fraction=0.025, pad=0.02)
    cbar.set_label(r"design base-shear coefficient  $V_b/W_t$")
    fig.suptitle(f"Site case-study CBF collapse fragilities by intensity measure "
                 f"({n}s, from design groups)", fontsize=13)
    fig_path = FIG_ROOT / f"{IDA_TAG}_collapse_fragilities_by_im_{n}s.png"
    fig.savefig(fig_path, dpi=200, bbox_inches="tight")
    print(f"saved {fig_path}")
    return fig


for n in STOREYS:
    if plot_fragilities(n) is not None:
        plt.show()

3s: nothing to plot
5s: nothing to plot


### Collapse median / dispersion vs design base-shear coefficient

Collapse median (in g) and dispersion $\beta$ against the design base-shear coefficient
$V_b/W_t$, one panel per intensity measure and one figure per storey count. Markers are
coloured by $V_b/W_t$ (same *viridis* scale) and joined in ascending order to show the
capacity trend.

In [17]:
def plot_param_vs_vbcoeff(n: int, param: str, ylabel: str, title: str):
    """3-panel scatter of a fragility parameter ('median'/'dispersion') vs Vb_coeff."""
    keys = [k for k in sorted(site_results) if k[0] == n]
    if not keys:
        print(f"{n}s: nothing to plot")
        return None
    vbs = np.array([VB_COEFF.get(k, np.nan) for k in keys])
    order = np.argsort(vbs)
    vbs_s = vbs[order]
    norm = Normalize(vmin=np.nanmin(vbs), vmax=np.nanmax(vbs))
    cmap = cm.viridis

    fig, axes = plt.subplots(1, 3, figsize=(13, 4.2), sharex=True)
    for ax, im in zip(axes, IM_ORDER):
        vals = np.array([site_results[k][im][param] for k in keys])[order]
        ax.plot(vbs_s, vals, "-", color="0.75", lw=1, zorder=1)
        ax.scatter(vbs_s, vals, c=vbs_s, cmap=cmap, norm=norm,
                   s=45, edgecolor="k", linewidths=0.5, zorder=2)
        ax.set_title(im, fontsize=11)
        ax.set_xlabel(r"$V_b/W_t$")
        ax.grid(True, lw=0.5, alpha=0.4)
    axes[0].set_ylabel(ylabel)
    fig.suptitle(f"{title} ({n}s)", fontsize=13)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    return fig


for n in STOREYS:
    fig_med = plot_param_vs_vbcoeff(
        n, "median", "collapse median [g]",
        "Site case-study CBF collapse median vs design base-shear coefficient")
    if fig_med is not None:
        fig_med.savefig(FIG_ROOT / f"collapse_median_vs_vbcoeff_{n}s.png",
                        dpi=200, bbox_inches="tight")
        plt.show()

    fig_disp = plot_param_vs_vbcoeff(
        n, "dispersion", r"collapse dispersion  $\beta$",
        "Site case-study CBF collapse dispersion vs design base-shear coefficient")
    if fig_disp is not None:
        fig_disp.savefig(FIG_ROOT / f"collapse_dispersion_vs_vbcoeff_{n}s.png",
                         dpi=200, bbox_inches="tight")
        plt.show()

3s: nothing to plot
3s: nothing to plot
5s: nothing to plot
5s: nothing to plot
